# Publication copy: portable paths and cleared outputs

Run the path configuration below before the original notebook cells. Change only PROJECT_ROOT. On Windows use a raw string such as r"D:\Phishing_Paper_Reproducibility". The data filenames and results/revision layout are preserved.

This copy has cleared cell outputs and execution counts. Keep the executed original and its saved results separately. This path revision is not a new scientific run or a full methodological audit.

**Historical preparation/audit notebook:** early cells include one-time data corrections and audits of the superseded run. It is not a safe top-to-bottom preparation recipe for already corrected inputs. Do not rerun those corrections. Some historical audits require older artifacts intentionally excluded from the publication package. The finalized analysis uses the supplied approved preparation files. Preparation regeneration needs a separate consolidation and verification step.


In [ ]:
# Publication path configuration: edit only PROJECT_ROOT.
from pathlib import Path, PurePosixPath
import sys

# Empty: infer the root when opened from the project root or notebooks/.
# Otherwise enter your local absolute folder, or a /content/drive/... folder.
PROJECT_ROOT = ""
if not PROJECT_ROOT:
    candidates = [Path.cwd(), Path.cwd().parent]
    found = [p for p in candidates if (p / "data").is_dir() and (p / "notebooks").is_dir()]
    if len(found) != 1:
        raise RuntimeError("Set PROJECT_ROOT to your Phishing_Paper_Reproducibility folder.")
    PROJECT_ROOT = found[0]
PROJECT_ROOT = Path(PROJECT_ROOT).expanduser()
if str(PROJECT_ROOT).startswith("/content/drive/"):
    if "google.colab" not in sys.modules:
        raise RuntimeError("A Drive root requires Colab. Use a local folder on your computer.")
    if not Path("/content/drive/MyDrive").is_dir():
        from google.colab import drive
        drive.mount("/content/drive")
PROJECT_ROOT = PROJECT_ROOT.resolve()
if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(f"Project root missing: {PROJECT_ROOT}")

DATA_FILENAMES = {
    "phiusiil.csv", "phiusiil_unified.csv", "URL-Phish.csv",
    "urlphish_unified.csv", "LegitPhish Dataset.csv", "legitphish_unified.csv",
}
def relocated_dataset_path(recorded_path):
    # Immutable historical records are kept byte-for-byte. Map only the
    # approved data members to this root; existing old Drive files are ignored.
    old = PurePosixPath(str(recorded_path).replace("\\", "/"))
    if old.parent.name != "data" or old.name not in DATA_FILENAMES:
        raise ValueError(f"Unrecognized recorded dataset: {recorded_path}")
    return PROJECT_ROOT / "data" / old.name

print("Project root:", PROJECT_ROOT)

def portable_recorded_path(value):
    """Relocate historical manifest paths in memory; never rewrite evidence."""
    old = PurePosixPath(str(value).replace("\\", "/"))
    if ".." in old.parts:
        raise ValueError("Unsafe recorded path")
    current = Path(value).expanduser()
    if current.is_absolute() and current.is_relative_to(PROJECT_ROOT):
        return current
    # Historical records are limited to the project data/results members.
    roots = [i for i, part in enumerate(old.parts) if part == "phishing_q1_project"]
    if len(roots) != 1:
        raise ValueError(f"Cannot relocate historical path: {value}")
    member = old.parts[roots[0]+1:]
    if not member or member[0] not in {"data", "results"}:
        raise ValueError(f"Unexpected historical member: {value}")
    return PROJECT_ROOT.joinpath(*member)


In [ ]:
# Intention: Robustly load datasets and dynamically harmonize target labels
import pandas as pd

# 1. Mount Google Drive securely
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
from collections import Counter
import csv
import os
import shutil
import tempfile

data_dir = Path(
    str(PROJECT_ROOT / 'data')
)

raw_path = data_dir / "LegitPhish Dataset.csv"
unified_path = data_dir / "legitphish_unified.csv"
backup_path = data_dir / "LegitPhish Dataset_before_missing_label_fix.csv"

missing_raw_index = 65118  # Zero-based data-row number; header is not counted.
missing_url = "https://www.chocolatleroux.com"

def read_csv(path):
    with path.open("r", encoding="utf-8-sig", newline="") as file:
        reader = csv.DictReader(file)
        return reader.fieldnames, list(reader)

raw_columns, raw_rows = read_csv(raw_path)
_, unified_rows = read_csv(unified_path)

if len(raw_rows) == 101218:
    print("Raw file already has 101,218 rows. No file was changed.")

else:
    # Verify the specific source problem.
    if len(raw_rows) != 101219:
        raise RuntimeError(f"Unexpected raw row count: {len(raw_rows)}")

    raw_labels = Counter(row["ClassLabel"] for row in raw_rows)
    if raw_labels != {"0": 63678, "1": 37540, "": 1}:
        raise RuntimeError(f"Unexpected raw labels: {raw_labels}")

    missing_row = raw_rows[missing_raw_index]
    if (
        missing_row["ClassLabel"] != ""
        or missing_row["URL"] != missing_url
    ):
        raise RuntimeError("Row 65,118 is not the expected unlabeled URL.")

    # Verify that the previously corrected unified file represents
    # these raw rows in the same order, skipping only the unlabeled row.
    if len(unified_rows) != 101218:
        raise RuntimeError(
            f"Expected 101,218 corrected unified rows; found {len(unified_rows)}"
        )

    for raw_index, raw_row in enumerate(raw_rows):
        if raw_index == missing_raw_index:
            continue

        unified_index = (
            raw_index if raw_index < missing_raw_index else raw_index - 1
        )
        unified_row = unified_rows[unified_index]

        expected_label = "1" if raw_row["ClassLabel"] == "0" else "0"
        expected_url_length = len(raw_row["URL"].strip())

        if (
            unified_row["label"] != expected_label
            or int(unified_row["URLLength"]) != expected_url_length
        ):
            raise RuntimeError(
                "Raw/unified alignment check failed at raw row "
                f"{raw_index}. No file was changed."
            )

    if backup_path.exists():
        raise RuntimeError(
            f"Backup already exists: {backup_path}. "
            "Inspect it before running this cell again."
        )

    corrected_rows = (
        raw_rows[:missing_raw_index] + raw_rows[missing_raw_index + 1:]
    )
    corrected_labels = Counter(
        row["ClassLabel"] for row in corrected_rows
    )
    if corrected_labels != {"0": 63678, "1": 37540}:
        raise RuntimeError(f"Unexpected corrected labels: {corrected_labels}")

    # Back up first; write a temporary file so a failed write cannot
    # leave a partially written raw dataset.
    shutil.copy2(raw_path, backup_path)
    temporary_path = None

    try:
        with tempfile.NamedTemporaryFile(
            mode="w",
            encoding="utf-8",
            newline="",
            dir=data_dir,
            prefix="legitphish_raw_fix_",
            suffix=".csv",
            delete=False,
        ) as file:
            temporary_path = Path(file.name)
            writer = csv.DictWriter(file, fieldnames=raw_columns)
            writer.writeheader()
            writer.writerows(corrected_rows)

        os.replace(temporary_path, raw_path)

    finally:
        if temporary_path is not None and temporary_path.exists():
            temporary_path.unlink()

    print("Raw backup:", backup_path)
    print("Corrected raw file:", raw_path)
    print("Rows:", len(corrected_rows))
    print("Raw labels (0=phishing, 1=legitimate):", dict(corrected_labels))
    print("Raw and unified row order checked.")

In [ ]:
from pathlib import Path
from collections import Counter
import hashlib

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

data_dir = Path(
    str(PROJECT_ROOT / 'data')
)

phi_raw = pd.read_csv(data_dir / "phiusiil.csv", keep_default_na=False)
phi_features = pd.read_csv(
    data_dir / "phiusiil_unified.csv", keep_default_na=False
)
legit_raw = pd.read_csv(
    data_dir / "LegitPhish Dataset.csv", keep_default_na=False
)
legit_features = pd.read_csv(
    data_dir / "legitphish_unified.csv", keep_default_na=False
)

# 1. Check counts and labels before using row positions.
assert len(phi_raw) == len(phi_features) == 235795
assert len(legit_raw) == len(legit_features) == 101218

assert set(phi_raw["label"].unique()) == {0, 1}
assert set(legit_raw["ClassLabel"].unique()) == {0, 1}
assert set(phi_features["label"].unique()) == {0, 1}
assert set(legit_features["label"].unique()) == {0, 1}

assert phi_raw["label"].value_counts().to_dict() == {0: 134850, 1: 100945}
assert legit_raw["ClassLabel"].value_counts().to_dict() == {
    0: 63678, 1: 37540
}

# Check the label and URL length for EVERY aligned raw/unified row.
assert np.array_equal(
    phi_raw["label"].to_numpy(),
    phi_features["label"].to_numpy()
)
assert np.array_equal(
    (1 - legit_raw["ClassLabel"]).to_numpy(),
    legit_features["label"].to_numpy()
)

for name, raw, features in [
    ("PhiUSIIL", phi_raw, phi_features),
    ("LegitPhish", legit_raw, legit_features),
]:
    if raw["URL"].isna().any() or raw["URL"].eq("").any():
        raise ValueError(f"{name} has an empty URL.")

    expected_lengths = raw["URL"].astype(str).str.strip().str.len()
    if not np.array_equal(
        expected_lengths.to_numpy(),
        features["URLLength"].to_numpy()
    ):
        raise ValueError(f"{name}: raw/unified row alignment failed.")

print("Raw/unified row alignment: passed for both datasets.")

# 2. Recreate the EXACT two-step source split in the revised notebook.
seed = 42
source_labels = phi_features["label"].to_numpy()
all_indices = np.arange(len(phi_features))

rest, source_test = train_test_split(
    all_indices,
    test_size=0.20,
    stratify=source_labels,
    random_state=seed,
)
source_train, source_validation = train_test_split(
    rest,
    test_size=0.20,
    stratify=source_labels[rest],
    random_state=seed,
)

assert (
    len(source_train),
    len(source_validation),
    len(source_test),
) == (150908, 37728, 47159)

print(
    "PhiUSIIL split:",
    len(source_train), "train;",
    len(source_validation), "validation;",
    len(source_test), "test"
)

# 3. Compare LITERAL URL strings. Do not normalize or change case here.
train_urls = set(phi_raw.iloc[source_train]["URL"].astype(str))
validation_urls = set(
    phi_raw.iloc[source_validation]["URL"].astype(str)
)
test_urls = set(phi_raw.iloc[source_test]["URL"].astype(str))

target_urls = legit_raw["URL"].astype(str)
in_train = target_urls.isin(train_urls)
in_validation = target_urls.isin(validation_urls)
in_test = target_urls.isin(test_urls)

# Native LegitPhish labels: 1=legitimate, 0=phishing.
for native_label, class_name in [(1, "legitimate"), (0, "phishing")]:
    class_rows = legit_raw["ClassLabel"].eq(native_label)
    print(f"\nLegitPhish {class_name}: {int(class_rows.sum()):,}")
    print("  Exact URLs in source training:",
          f"{int((class_rows & in_train).sum()):,}")
    print("  Exact URLs in source validation:",
          f"{int((class_rows & in_validation).sum()):,}")
    print("  Exact URLs in source test:",
          f"{int((class_rows & in_test).sum()):,}")

print("\nRows eligible for training-URL-disjoint LegitPhish evaluation:")
print("  Total:", f"{int((~in_train).sum()):,}")
print(
    "  Legitimate:",
    f"{int((~in_train & legit_raw['ClassLabel'].eq(1)).sum()):,}"
)
print(
    "  Phishing:",
    f"{int((~in_train & legit_raw['ClassLabel'].eq(0)).sum()):,}"
)

# Flag label conflicts if a URL appears in both source training
# and LegitPhish with different native class meanings.
source_url_labels = (
    phi_raw.iloc[source_train]
    .groupby("URL")["label"]
    .agg(lambda values: set(values))
)
conflicting_source_urls = {
    url for url, values in source_url_labels.items() if len(values) > 1
}
if conflicting_source_urls:
    raise ValueError(
        f"{len(conflicting_source_urls)} source-training URLs "
        "have conflicting labels within PhiUSIIL."
    )

matched_target = legit_raw.loc[in_train, ["URL", "ClassLabel"]].copy()
if len(matched_target):
    source_label_for_match = matched_target["URL"].map(
        source_url_labels.map(lambda values: next(iter(values)))
    )
    expected_harmonized = 1 - matched_target["ClassLabel"]
    disagreements = int(
        (source_label_for_match.to_numpy()
         != expected_harmonized.to_numpy()).sum()
    )
else:
    disagreements = 0

print("\nConflicting class labels among training URL matches:",
      disagreements)

# A fingerprint of the ordered training-row indices helps identify
# the exact reconstructed split when we create the cohort manifest.
print(
    "Training-index SHA256:",
    hashlib.sha256(
        np.asarray(source_train, dtype="<i8").tobytes()
    ).hexdigest()
)

In [ ]:
# Uses legit_raw, in_train, in_validation, and in_test
# from the previous read-only audit cell.
required = ["legit_raw", "in_train", "in_validation", "in_test"]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        f"Run the previous overlap-audit cell first. Missing: {missing}"
    )

in_development = in_train | in_validation
in_any_source = in_train | in_validation | in_test

for name, keep in [
    ("Absent from source training", ~in_train),
    ("Absent from source training AND validation", ~in_development),
    ("Absent from ALL source partitions", ~in_any_source),
]:
    legitimate = int((keep & legit_raw["ClassLabel"].eq(1)).sum())
    phishing = int((keep & legit_raw["ClassLabel"].eq(0)).sum())
    print(f"{name}:")
    print(
        f"  {legitimate:,} legitimate + {phishing:,} phishing "
        f"= {int(keep.sum()):,} total"
    )

# Show whether the three reported overlap counts describe
# different rows or the same rows appearing in multiple partitions.
patterns = (
    pd.DataFrame({
        "train": in_train.to_numpy(),
        "validation": in_validation.to_numpy(),
        "test": in_test.to_numpy(),
        "native_label": legit_raw["ClassLabel"].to_numpy(),
    })
    .groupby(
        ["native_label", "train", "validation", "test"],
        dropna=False
    )
    .size()
    .reset_index(name="rows")
)

print("\nOverlap combinations (native label: 1=legitimate, 0=phishing):")
display(patterns)

In [ ]:
from pathlib import Path
import csv
import hashlib

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

data_dir = Path(
    str(PROJECT_ROOT / 'data')
)

source_raw_path = data_dir / "phiusiil.csv"
source_unified_path = data_dir / "phiusiil_unified.csv"
target_raw_path = data_dir / "LegitPhish Dataset.csv"
target_unified_path = data_dir / "legitphish_unified.csv"

cohort_path = data_dir / "legitphish_unified_development_disjoint.csv"
manifest_path = data_dir / "legitphish_development_disjoint_rows.csv"

# Never overwrite an existing cohort or its row record.
for output_path in (cohort_path, manifest_path):
    if output_path.exists():
        raise FileExistsError(
            f"{output_path} already exists. Inspect it before rerunning."
        )

source_raw = pd.read_csv(source_raw_path, keep_default_na=False)
source_unified = pd.read_csv(source_unified_path, keep_default_na=False)
target_raw = pd.read_csv(target_raw_path, keep_default_na=False)
target_unified = pd.read_csv(target_unified_path, keep_default_na=False)

# Recheck that row positions still refer to the expected records.
if len(source_raw) != len(source_unified) or len(source_raw) != 235795:
    raise RuntimeError("PhiUSIIL raw/unified row counts do not match.")
if len(target_raw) != len(target_unified) or len(target_raw) != 101218:
    raise RuntimeError("LegitPhish raw/unified row counts do not match.")

if not np.array_equal(
    source_raw["label"].to_numpy(),
    source_unified["label"].to_numpy()
):
    raise RuntimeError("PhiUSIIL labels are not row-aligned.")

if not np.array_equal(
    (1 - target_raw["ClassLabel"]).to_numpy(),
    target_unified["label"].to_numpy()
):
    raise RuntimeError("LegitPhish labels are not row-aligned.")

for name, raw, unified in (
    ("PhiUSIIL", source_raw, source_unified),
    ("LegitPhish", target_raw, target_unified),
):
    if not np.array_equal(
        raw["URL"].astype(str).str.strip().str.len().to_numpy(),
        unified["URLLength"].to_numpy()
    ):
        raise RuntimeError(f"{name} URL lengths are not row-aligned.")

# Recreate the exact source split used in the revised pipeline.
labels = source_unified["label"].to_numpy()
rest, source_test = train_test_split(
    np.arange(len(labels)),
    test_size=0.20,
    stratify=labels,
    random_state=42,
)
source_train, source_validation = train_test_split(
    rest,
    test_size=0.20,
    stratify=labels[rest],
    random_state=42,
)

assert (len(source_train), len(source_validation), len(source_test)) == (
    150908, 37728, 47159
)

split_hash = hashlib.sha256(
    np.asarray(source_train, dtype="<i8").tobytes()
).hexdigest()
expected_hash = (
    "e837161f39158037e6e7821fbcb805"
    "5d242585d5a072e47699984905f41342eb"
)
if split_hash != expected_hash:
    raise RuntimeError("Source training split differs from the audited split.")

# Exact comparison of literal URL strings; no URL normalization.
train_urls = set(source_raw.iloc[source_train]["URL"].astype(str))
validation_urls = set(
    source_raw.iloc[source_validation]["URL"].astype(str)
)
test_urls = set(source_raw.iloc[source_test]["URL"].astype(str))

urls = target_raw["URL"].astype(str)
in_train = urls.isin(train_urls)
in_validation = urls.isin(validation_urls)
in_test = urls.isin(test_urls)

keep = ~(in_train | in_validation)

# Check the audited class counts and absence of development URL matches.
kept_labels = target_unified.loc[keep, "label"].value_counts().to_dict()
if len(target_unified.loc[keep]) != 71619 or kept_labels != {
    0: 7941, 1: 63678
}:
    raise RuntimeError(
        f"Unexpected cohort size or labels: "
        f"{int(keep.sum())} rows; {kept_labels}"
    )

if (in_train & keep).any() or (in_validation & keep).any():
    raise RuntimeError("A source-development URL remains in the cohort.")

# Keep the exact existing feature columns and their original order.
cohort = target_unified.loc[keep].reset_index(drop=True)
if not cohort.equals(
    target_unified.iloc[np.flatnonzero(keep.to_numpy())].reset_index(drop=True)
):
    raise RuntimeError("Feature row order changed.")

# This record makes the filtering reproducible. Row numbers are zero-based
# data-row positions AFTER removal of the one unlabeled raw record.
original_indices = np.flatnonzero(keep.to_numpy())
row_record = pd.DataFrame({
    "cohort_row_index": np.arange(len(original_indices)),
    "corrected_raw_and_unified_row_index": original_indices,
    "matched_source_test_url": in_test.iloc[original_indices]
        .to_numpy(dtype=bool),
    "harmonized_label": cohort["label"].to_numpy(),
})

if int(row_record["matched_source_test_url"].sum()) != 7336:
    raise RuntimeError(
        "Unexpected number of remaining source-test URL matches."
    )

# Create new files only after all checks have passed.
cohort.to_csv(cohort_path, index=False)
row_record.to_csv(manifest_path, index=False)

print("Cohort:", cohort_path)
print("Row record:", manifest_path)
print("Rows:", f"{len(cohort):,}")
print("Labels (0=legitimate, 1=phishing):", kept_labels)
print(
    "Remaining exact URL matches to source test:",
    f"{int(row_record['matched_source_test_url'].sum()):,}"
)
print("Audited training-index SHA256:", split_hash)

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import numpy as np
import pandas as pd

RUN_DIR = Path(
    str(PROJECT_ROOT / 'results/revision/full_20261020T193941835689Z')
)


def file_sha256(path):
    path = portable_recorded_path(path) if "phishing_q1_project" in str(path) else Path(path)
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def overlap_counts(source, target):
    """Count shared URL identities and matching rows separately."""
    shared = set(source["url"]) & set(target["url"])
    source_matches = source["url"].isin(shared)
    target_matches = target["url"].isin(shared)

    conflicts = (
        (
            set(source.loc[source.label.eq(0), "url"])
            & set(target.loc[target.label.eq(1), "url"])
        )
        | (
            set(source.loc[source.label.eq(1), "url"])
            & set(target.loc[target.label.eq(0), "url"])
        )
    )

    return {
        "source_rows": len(source),
        "target_rows": len(target),
        "shared_unique_urls": len(shared),
        "source_rows_with_shared_url": int(source_matches.sum()),
        "target_rows_with_shared_url": int(target_matches.sum()),
        "target_legitimate_rows_with_shared_url": int(
            (target_matches & target.label.eq(0)).sum()
        ),
        "target_phishing_rows_with_shared_url": int(
            (target_matches & target.label.eq(1)).sum()
        ),
        "shared_urls_with_conflicting_labels": len(conflicts),
    }


def run_reference_overlap_audit(run_dir):
    run_dir = Path(run_dir)
    manifest_path = run_dir / "run_manifest.json"

    if not manifest_path.is_file():
        raise FileNotFoundError(
            f"Completed run not found: {manifest_path}. "
            "Mount Google Drive and check RUN_DIR."
        )

    manifest = json.loads(manifest_path.read_text())

    if (
        manifest.get("status") != "complete"
        or manifest.get("mode") != "full"
    ):
        raise ValueError("This audit requires a completed full run.")

    inventory = pd.read_csv(
        run_dir / "artifact_inventory.csv"
    ).set_index("path")

    verified_payloads = []

    def checked_payload(relative):
        """Verify an existing run artifact before reading it."""
        if relative not in inventory.index:
            raise ValueError(f"Run inventory has no entry for {relative}")

        path = run_dir / relative
        row = inventory.loc[relative]

        if (
            not path.is_file()
            or path.stat().st_size != int(row["bytes"])
            or file_sha256(path) != row["sha256"]
        ):
            raise ValueError(
                f"Completed-run artifact is missing or changed: {relative}"
            )

        verified_payloads.append(relative)
        return path

    # Use the exact raw files and row record recorded by the completed run.
    provenance = manifest["cohort_provenance_files"]
    inputs = {
        name: provenance[name]
        for name in ["source_raw", "corrected_raw", "row_record"]
    }

    for name, spec in inputs.items():
        if (
            not Path(portable_recorded_path(spec["path"])).is_file()
            or file_sha256(portable_recorded_path(spec["path"])) != spec["sha256"]
        ):
            raise ValueError(
                f"{name} differs from the completed run: {portable_recorded_path(spec['path'])}"
            )

    raw_source = pd.read_csv(
        portable_recorded_path(inputs["source_raw"]["path"]),
        usecols=["URL", "label"],
        low_memory=False,
    )
    raw_target = pd.read_csv(
        portable_recorded_path(inputs["corrected_raw"]["path"]),
        usecols=["URL", "ClassLabel"],
        low_memory=False,
    )
    record = pd.read_csv(portable_recorded_path(inputs["row_record"]["path"]))

    saved_record = pd.read_csv(
        checked_payload(
            "splits/legitphish_development_disjoint_rows.csv.gz"
        )
    )
    pd.testing.assert_frame_equal(
        record, saved_record, check_dtype=False
    )

    if len(raw_source) != manifest["input_files"]["PhiUSIIL"]["rows"]:
        raise ValueError("Raw source row count differs from the manifest.")

    expected = manifest["legitphish_cohort_audit"]

    # The audited release has mutually exclusive training/validation matches.
    expected_raw_rows = (
        expected["cohort_rows"]
        + expected["excluded_train_matches"]
        + expected["excluded_validation_matches"]
    )

    if (
        len(raw_target) != expected_raw_rows
        or not raw_target.ClassLabel.isin([0, 1]).all()
    ):
        raise ValueError(
            "Corrected raw target has an unexpected row count "
            "or invalid labels."
        )

    if not raw_source.label.isin([0, 1]).all():
        raise ValueError("Raw source has invalid labels.")

    if not np.array_equal(
        record.cohort_row_index, np.arange(len(record))
    ):
        raise ValueError(
            "Cohort positions are not consecutive and ordered."
        )

    positions = record.corrected_raw_and_unified_row_index.to_numpy()

    if (
        positions.dtype.kind not in "iu"
        or len(np.unique(positions)) != len(positions)
    ):
        raise ValueError("Cohort raw positions must be distinct integers.")

    if (positions < 0).any() or (positions >= len(raw_target)).any():
        raise ValueError("Cohort raw position is out of range.")

    # Preserve the exact URL-matching rule used to create the cohort.
    def normalized(values):
        return values.fillna("").astype(str).str.strip()

    source = pd.DataFrame({
        "url": normalized(raw_source.URL),
        "label": raw_source.label.astype(int),
    })
    source.index = [f"PhiUSIIL:{i}" for i in range(len(source))]

    target = pd.DataFrame(
        {
            "url": normalized(
                raw_target.iloc[positions].URL
            ).to_numpy(),
            # Native LegitPhish: 1=legitimate, 0=phishing.
            "label": (
                1 - raw_target.iloc[positions].ClassLabel
            ).to_numpy(dtype=int),
        },
        index=[f"LegitPhish:{i}" for i in positions],
    )

    if source.url.eq("").any() or target.url.eq("").any():
        raise ValueError(
            "Blank URLs prevent an unambiguous identity audit."
        )

    if (
        len(target) != expected["cohort_rows"]
        or int(target.label.eq(0).sum()) != expected["legitimate"]
        or int(target.label.eq(1).sum()) != expected["phishing"]
        or not np.array_equal(
            target.label, record.harmonized_label
        )
    ):
        raise ValueError(
            "Cohort labels or counts differ from the completed run."
        )

    # Recover the original source partitions from their saved row IDs.
    split = pd.read_csv(
        checked_payload(
            "splits/source_train_validation_test.csv.gz"
        )
    )

    if (
        not split.row_id.is_unique
        or set(split.row_id) != set(source.index)
        or set(split.partition) != {"train", "validation", "test"}
    ):
        raise ValueError(
            "Saved source partition is not a complete row partition."
        )

    if not np.array_equal(
        source.loc[split.row_id, "label"], split.label
    ):
        raise ValueError(
            "Saved source labels do not align with raw source rows."
        )

    source_csv_rows = pd.Series(
        np.arange(len(source)) + 2,
        index=source.index,
    )

    if not np.array_equal(
        source_csv_rows.loc[split.row_id], split.raw_csv_row
    ):
        raise ValueError(
            "Saved source IDs do not match their raw CSV positions."
        )

    source_pools = {
        part: source.loc[
            split.loc[split.partition.eq(part), "row_id"]
        ]
        for part in ["train", "validation", "test"]
    }

    reference_rows = [
        dict(
            reference=f"source_{part}",
            **overlap_counts(block, target),
        )
        for part, block in source_pools.items()
    ]

    if any(
        row["target_rows_with_shared_url"]
        for row in reference_rows[:2]
    ):
        raise ValueError(
            "Cohort unexpectedly overlaps source training or validation."
        )

    if (
        reference_rows[2]["target_rows_with_shared_url"]
        != expected["retained_source_test_matches"]
    ):
        raise ValueError(
            "Source-test overlap differs from the manifest."
        )

    # Check the actual saved MMD samples, not just their full sampling pools.
    files = sorted(
        (run_dir / "mmd").glob("LegitPhish_*_ids.npz")
    )

    expected_files = 4 * (manifest["config"]["mmd_repeats"] + 1)
    if len(files) != expected_files:
        raise ValueError(
            "Expected every LegitPhish MMD repeat "
            "and inference sample ID file."
        )

    test_ids = set(source_pools["test"].index)
    sample_rows = []

    for path in files:
        checked_payload(str(path.relative_to(run_dir)))

        with np.load(path, allow_pickle=False) as saved:
            source_ids = saved["source_ids"].astype(str)
            target_ids = saved["target_ids"].astype(str)

        if (
            len(set(source_ids)) != len(source_ids)
            or len(set(target_ids)) != len(target_ids)
            or not set(source_ids).issubset(test_ids)
            or not set(target_ids).issubset(target.index)
        ):
            raise ValueError(
                f"Invalid saved sample row identities: {path.name}"
            )

        stem = (
            path.name
            .removeprefix("LegitPhish_")
            .removesuffix("_ids.npz")
        )

        if stem.endswith("_inference"):
            comparison = stem.removesuffix("_inference")
            kind = "inference"
        else:
            comparison, _ = stem.rsplit("_repeat", 1)
            kind = "stability_repeat"

        sample_rows.append(
            dict(
                sample=path.name,
                comparison=comparison,
                draw_type=kind,
                **overlap_counts(
                    source.loc[source_ids],
                    target.loc[target_ids],
                ),
            )
        )

    # Check both source-test pools used by the entropy stage.
    entropy_split = pd.read_csv(
        checked_payload(
            "splits/entropy_source_test_null_and_evaluation.csv.gz"
        )
    )

    if (
        not entropy_split.row_id.is_unique
        or set(entropy_split.row_id) != test_ids
        or set(entropy_split.partition)
        != {
            "null_threshold_pool",
            "source_false_alarm_evaluation",
        }
    ):
        raise ValueError(
            "Entropy pools do not partition the saved source-test rows."
        )

    for part, block in entropy_split.groupby("partition"):
        reference_rows.append(
            dict(
                reference=f"entropy_{part}",
                **overlap_counts(
                    source.loc[block.row_id],
                    target,
                ),
            )
        )

    detailed = pd.DataFrame(sample_rows)
    summary = detailed.groupby(
        ["comparison", "draw_type"],
        as_index=False,
    ).agg(
        samples=("sample", "size"),
        samples_with_overlap=(
            "shared_unique_urls",
            lambda values: int(values.gt(0).sum()),
        ),
        shared_urls_min=("shared_unique_urls", "min"),
        shared_urls_mean=("shared_unique_urls", "mean"),
        shared_urls_max=("shared_unique_urls", "max"),
    )

    duplicates = []
    for label, group in target.groupby("label"):
        duplicates.append({
            "label": int(label),
            "rows": len(group),
            "unique_urls": group.url.nunique(),
            "rows_beyond_first_occurrence": int(
                group.url.duplicated().sum()
            ),
        })

    # Ensure raw inputs stayed unchanged while the audit was running.
    for name, spec in inputs.items():
        if file_sha256(portable_recorded_path(spec["path"])) != spec["sha256"]:
            raise ValueError(f"{name} changed during the audit.")

    cross_labels = len(
        set(target.loc[target.label.eq(0), "url"])
        & set(target.loc[target.label.eq(1), "url"])
    )

    # Save only aggregate counts into a separate folder.
    stamp = datetime.now(timezone.utc).strftime(
        "%Y%m%dT%H%M%S%fZ"
    )
    out = run_dir.parent / (
        f"{run_dir.name}_reference_overlap_audit_{stamp}"
    )
    out.mkdir()

    pd.DataFrame(reference_rows).to_csv(
        out / "reference_pool_overlap.csv", index=False
    )
    detailed.to_csv(
        out / "MMD_sample_overlap.csv", index=False
    )
    summary.to_csv(
        out / "MMD_overlap_summary.csv", index=False
    )
    pd.DataFrame(duplicates).to_csv(
        out / "cohort_duplicate_summary.csv", index=False
    )

    (out / "audit_manifest.json").write_text(
        json.dumps(
            {
                "parent_run": str(run_dir),
                "parent_manifest_sha256": file_sha256(
                    manifest_path
                ),
                "normalization": "strip surrounding whitespace only",
                "verified_input_hashes": inputs,
                "verified_run_payloads": verified_payloads,
                "cohort_urls_shared_between_labels": cross_labels,
                "scope": (
                    "LegitPhish exact URL identity only; "
                    "domain dependence and URL-Phish "
                    "are not audited here"
                ),
                "interpretation": (
                    "Identity overlap counts only; no automatic "
                    "conclusion about bias direction "
                    "or corrected inference"
                ),
            },
            indent=2,
        ) + "\n"
    )

    print(
        "\nREFERENCE POOLS\n"
        + pd.DataFrame(reference_rows).to_string(index=False)
    )
    print(
        "\nMMD SAMPLE SUMMARY\n"
        + summary.to_string(index=False)
    )
    print(
        "\nCOHORT DUPLICATES\n"
        + pd.DataFrame(duplicates).to_string(index=False)
    )
    print("\nCohort URLs carrying both labels:", cross_labels)
    print("\nSeparate audit folder:", out)

    return out


AUDIT_OUTPUT = run_reference_overlap_audit(RUN_DIR)

In [ ]:
from pathlib import Path
from itertools import combinations
import hashlib
import json
import numpy as np
import pandas as pd


def audit_saved_url_splits(run_dir, output_dir):
    run_dir = Path(run_dir)
    output_dir = Path(output_dir)

    manifest = json.loads(
        (run_dir / "run_manifest.json").read_text()
    )
    if (
        manifest.get("status") != "complete"
        or manifest.get("mode") != "full"
    ):
        raise ValueError("A completed full run is required.")

    inventory = pd.read_csv(
        run_dir / "artifact_inventory.csv"
    ).set_index("path")

    def digest(path):
        path = portable_recorded_path(path) if "phishing_q1_project" in str(path) else Path(path)
        h = hashlib.sha256()
        with Path(path).open("rb") as handle:
            for chunk in iter(
                lambda: handle.read(1048576), b""
            ):
                h.update(chunk)
        return h.hexdigest()

    def checked(relative):
        path = run_dir / relative
        if relative not in inventory.index or not path.is_file():
            raise ValueError(
                f"Missing recorded artifact: {relative}"
            )
        row = inventory.loc[relative]
        if (
            path.stat().st_size != int(row["bytes"])
            or digest(path) != row["sha256"]
        ):
            raise ValueError(
                f"Changed recorded artifact: {relative}"
            )
        return path

    inputs = {
        key: manifest["cohort_provenance_files"][key]
        for key in ["source_raw", "corrected_raw", "row_record"]
    }

    for key, spec in inputs.items():
        if (
            not Path(portable_recorded_path(spec["path"])).is_file()
            or digest(portable_recorded_path(spec["path"])) != spec["sha256"]
        ):
            raise ValueError(
                f"Raw input differs from completed run: {key}"
            )

    raw = pd.read_csv(
        portable_recorded_path(inputs["corrected_raw"]["path"]),
        usecols=["URL", "ClassLabel"],
    )
    record = pd.read_csv(portable_recorded_path(inputs["row_record"]["path"]))
    positions = (
        record.corrected_raw_and_unified_row_index.to_numpy()
    )

    if (
        positions.dtype.kind not in "iu"
        or len(set(positions)) != len(positions)
    ):
        raise ValueError("Invalid corrected raw row positions.")

    if (
        (positions < 0).any()
        or (positions >= len(raw)).any()
    ):
        raise ValueError(
            "Corrected raw row position is out of bounds."
        )

    if not raw.ClassLabel.isin([0, 1]).all():
        raise ValueError(
            "Corrected target raw labels are invalid."
        )

    target = pd.DataFrame(
        {
            "url": (
                raw.iloc[positions].URL
                .fillna("")
                .astype(str)
                .str.strip()
                .to_numpy()
            ),
            "label": (
                1 - raw.iloc[positions].ClassLabel
            ).to_numpy(dtype=int),
        },
        index=[f"LegitPhish:{i}" for i in positions],
    )

    if (
        target.url.eq("").any()
        or not np.array_equal(
            target.label, record.harmonized_label
        )
    ):
        raise ValueError(
            "Target URLs or row-record labels are invalid."
        )

    repeats = manifest["config"]["repeats"]

    calibration_files = sorted(
        (run_dir / "splits").glob(
            "calibration_LegitPhish_*.csv.gz"
        )
    )
    budget_files = sorted(
        (run_dir / "splits").glob(
            "budget_LegitPhish_*.csv.gz"
        )
    )

    if (
        len(calibration_files) != repeats
        or len(budget_files)
        != repeats
        * len(manifest["config"]["budget_fractions"])
    ):
        raise ValueError(
            "Not all recorded calibration/budget splits "
            "are present."
        )

    rows = []

    for path in calibration_files + budget_files:
        d = pd.read_csv(
            checked(str(path.relative_to(run_dir)))
        )
        budget = path.name.startswith("budget_")

        if budget:
            a_name = "labeled_adaptation"
            b_name = "untouched_evaluation"
        else:
            a_name = "adaptation"
            b_name = "evaluation"

        if (
            set(d.partition) != {a_name, b_name}
            or not d.row_id.is_unique
            or not set(d.row_id).issubset(target.index)
        ):
            raise ValueError(
                f"Invalid row membership: {path.name}"
            )

        if not np.array_equal(
            target.loc[d.row_id, "label"], d.label
        ):
            raise ValueError(
                f"Labels do not align: {path.name}"
            )

        a = target.loc[
            d.loc[d.partition.eq(a_name), "row_id"]
        ]
        b = target.loc[
            d.loc[d.partition.eq(b_name), "row_id"]
        ]

        shared = set(a.url) & set(b.url)

        tail = (
            path.name.removesuffix(".csv.gz")
            .split("LegitPhish_", 1)[1]
        )
        repeat = int(tail.rsplit("_", 1)[-1])
        fraction = (
            float(tail.rsplit("_", 1)[0])
            if budget else 0.5
        )

        seen = b.url.isin(shared)

        rows.append({
            "experiment": (
                "label_budget" if budget else "calibration"
            ),
            "budget_fraction": fraction,
            "repeat": repeat,
            "labeled_rows": len(a),
            "labeled_unique_urls": a.url.nunique(),
            "shared_unique_urls": len(shared),
            "evaluation_rows_with_seen_url": int(
                seen.sum()
            ),
            "legitimate_evaluation_rows_with_seen_url": int(
                (seen & b.label.eq(0)).sum()
            ),
            "phishing_evaluation_rows_with_seen_url": int(
                (seen & b.label.eq(1)).sum()
            ),
        })

    detail = pd.DataFrame(rows)

    summary = detail.groupby(
        ["experiment", "budget_fraction"],
        as_index=False,
    ).agg(
        draws=("repeat", "size"),
        draws_with_overlap=(
            "shared_unique_urls",
            lambda values: int(values.gt(0).sum()),
        ),
        shared_urls_min=("shared_unique_urls", "min"),
        shared_urls_mean=("shared_unique_urls", "mean"),
        shared_urls_max=("shared_unique_urls", "max"),
        evaluation_rows_with_seen_url_mean=(
            "evaluation_rows_with_seen_url", "mean"
        ),
    )

    # Check source train/validation/test URL separation too.
    raw_source = pd.read_csv(
        portable_recorded_path(inputs["source_raw"]["path"]),
        usecols=["URL", "label"],
    )
    source = pd.DataFrame({
        "url": (
            raw_source.URL.fillna("")
            .astype(str).str.strip()
        ),
        "label": raw_source.label,
    })
    source.index = [
        f"PhiUSIIL:{i}" for i in range(len(source))
    ]

    split = pd.read_csv(
        checked(
            "splits/source_train_validation_test.csv.gz"
        )
    )

    if (
        not split.row_id.is_unique
        or set(split.row_id) != set(source.index)
        or set(split.partition)
        != {"train", "validation", "test"}
        or source.url.eq("").any()
    ):
        raise ValueError(
            "Invalid source identities or source split."
        )

    if not np.array_equal(
        source.loc[split.row_id, "label"], split.label
    ):
        raise ValueError(
            "Source labels do not align with saved splits."
        )

    pools = {
        part: source.loc[block.row_id]
        for part, block in split.groupby("partition")
    }

    source_rows = []

    for first, second in combinations(
        ["train", "validation", "test"], 2
    ):
        shared = (
            set(pools[first].url)
            & set(pools[second].url)
        )

        source_rows.append({
            "first_partition": first,
            "second_partition": second,
            "shared_unique_urls": len(shared),
            "first_rows_with_shared_url": int(
                pools[first].url.isin(shared).sum()
            ),
            "second_rows_with_shared_url": int(
                pools[second].url.isin(shared).sum()
            ),
        })

    for key, spec in inputs.items():
        if digest(portable_recorded_path(spec["path"])) != spec["sha256"]:
            raise ValueError(
                f"Input changed during audit: {key}"
            )

    if (
        output_dir.resolve() == run_dir.resolve()
        or run_dir.resolve()
        in output_dir.resolve().parents
    ):
        raise ValueError(
            "Use the separate audit folder, "
            "outside the completed run."
        )

    output_dir.mkdir(parents=True, exist_ok=True)

    detail.to_csv(
        output_dir / "calibration_budget_URL_overlap.csv",
        index=False,
    )
    summary.to_csv(
        output_dir
        / "calibration_budget_URL_overlap_summary.csv",
        index=False,
    )
    pd.DataFrame(source_rows).to_csv(
        output_dir / "source_partition_URL_overlap.csv",
        index=False,
    )

    print(
        "SOURCE PARTITIONS\n"
        + pd.DataFrame(source_rows).to_string(index=False)
    )
    print(
        "\nCALIBRATION AND LABEL BUDGET SPLITS\n"
        + summary.to_string(index=False)
    )
    print("\nSaved count tables:", output_dir)

    return summary


if "RUN_DIR" not in globals() or "AUDIT_OUTPUT" not in globals():
    raise RuntimeError(
        "Run the previous reference-overlap audit cell first."
    )

SPLIT_AUDIT_SUMMARY = audit_saved_url_splits(
    RUN_DIR, AUDIT_OUTPUT
)

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

# Use the completed run already identified in the earlier audit.
manifest = json.loads(
    (Path(RUN_DIR) / "run_manifest.json").read_text()
)

raw_path = portable_recorded_path(manifest["cohort_provenance_files"]["source_raw"]["path"])
unified_path = portable_recorded_path(manifest["input_files"]["PhiUSIIL"]["path"])

raw = pd.read_csv(raw_path, usecols=["URL", "label"])
unified = pd.read_csv(
    unified_path,
    keep_default_na=False,
    dtype={"TLD": "string"}
)

# Check basic alignment before comparing corresponding rows.
assert len(raw) == len(unified), "Raw/unified row counts differ."
assert raw["label"].isin([0, 1]).all(), "Unexpected source labels."
assert np.array_equal(
    raw["label"].to_numpy(),
    unified["label"].to_numpy()
), "Raw/unified labels are not aligned."

urls = raw["URL"].fillna("").astype(str).str.strip()
assert not urls.eq("").any(), "Blank source URLs need inspection."

repeated = urls.duplicated(keep=False)

label_variants = (
    raw.loc[repeated, ["label"]]
    .groupby(urls[repeated])["label"]
    .nunique()
)

feature_variants = (
    unified.loc[repeated]
    .drop(columns="label")
    .groupby(urls[repeated])
    .nunique(dropna=False)
)

report = pd.DataFrame({
    "check": [
        "Source rows",
        "Distinct URLs",
        "URLs appearing more than once",
        "Extra repeated rows",
        "URLs with conflicting labels",
        "URLs with different feature values",
    ],
    "count": [
        len(raw),
        urls.nunique(),
        urls[repeated].nunique(),
        int(urls.duplicated().sum()),
        int(label_variants.gt(1).sum()),
        int(feature_variants.gt(1).any(axis=1).sum()),
    ]
})

print(report.to_string(index=False))

varying_columns = feature_variants.columns[
    feature_variants.gt(1).any(axis=0)
].tolist()

print("\nFeature columns differing within repeated URLs:")
print(varying_columns or "None")

In [ ]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path(
    str(PROJECT_ROOT / 'data')
)

# Mount Drive only if the project folder is unavailable.
if not DATA_DIR.is_dir():
        drive.mount("/content/drive")

if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Dataset folder not found: {DATA_DIR}")

records = []

for path in sorted(DATA_DIR.glob("*.csv")):
    try:
        columns = pd.read_csv(path, nrows=0).columns.tolist()

        # Identify URL columns without printing any actual URLs.
        url_columns = [
            col for col in columns
            if str(col).strip().lower().replace("_", "").replace(" ", "")
            in {"url", "urls", "urladdress", "website", "webpage"}
        ]

        label_columns = [
            col for col in columns
            if str(col).strip().lower().replace("_", "").replace(" ", "")
            in {"label", "labels", "class", "classlabel", "target", "status"}
        ]

        if url_columns:
            records.append({
                "filename": path.name,
                "URL columns": ", ".join(url_columns),
                "label columns": ", ".join(label_columns) or "Not identified",
                "total columns": len(columns),
            })

    except Exception as error:
        print(f"Could not inspect {path.name}: {error}")

if records:
    print(pd.DataFrame(records).to_string(index=False))
else:
    print("No CSV with a recognizable URL column was found.")

print("\nNo dataset files were changed.")

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA_DIR = Path(
    str(PROJECT_ROOT / 'data')
)

raw = pd.read_csv(
    DATA_DIR / "URL-Phish.csv",
    usecols=["url", "label"],
    keep_default_na=False
)

unified = pd.read_csv(
    DATA_DIR / "urlphish_unified.csv",
    keep_default_na=False,
    dtype={"TLD": "string"}
)

# Both URL-Phish files should use 0=legitimate, 1=phishing.
# Validate explicitly; do not guess or automatically reverse labels.
raw_labels = pd.to_numeric(raw["label"], errors="raise")
unified_labels = pd.to_numeric(unified["label"], errors="raise")

if not raw_labels.isin([0, 1]).all():
    raise ValueError("Original URL-Phish contains missing or unexpected labels.")

if not unified_labels.isin([0, 1]).all():
    raise ValueError("Unified URL-Phish contains missing or unexpected labels.")

print("ROW COUNTS")
print("Original:", len(raw))
print("Unified: ", len(unified))

print("\nLABEL COUNTS — 0=legitimate, 1=phishing")
print("Original:", raw_labels.value_counts().sort_index().to_dict())
print("Unified: ", unified_labels.value_counts().sort_index().to_dict())

if len(raw) != len(unified):
    raise ValueError("Row counts differ. Stop before assigning URL identities.")

label_mismatches = int(np.count_nonzero(
    raw_labels.to_numpy() != unified_labels.to_numpy()
))
print("\nRows with different labels:", label_mismatches)

# Use the same URL comparison rule as our earlier audits:
# remove surrounding whitespace only.
original_urls = raw["url"].astype(str)
urls = original_urls.str.strip()
nonblank = urls.ne("")

counts = urls[nonblank].value_counts()
label_variants = raw_labels[nonblank].groupby(urls[nonblank]).nunique()

print("\nURL CHECK")
print("Blank URL rows:", int((~nonblank).sum()))
print("Distinct nonblank URLs:", int(counts.size))
print("URLs appearing more than once:", int(counts.gt(1).sum()))
print("Extra repeated rows:", int(counts.sub(1).sum()))
print("URLs with conflicting labels:", int(label_variants.gt(1).sum()))

# URL length provides another row-order check where available.
# Matching labels and lengths support alignment but do not prove URL identity.
if "URLLength" in unified.columns:
    lengths = pd.to_numeric(unified["URLLength"], errors="raise").to_numpy()

    print("\nURL LENGTH CHECK")
    print("Rows differing from original URL length:",
          int(np.count_nonzero(lengths != original_urls.str.len().to_numpy())))
    print("Rows differing from trimmed URL length:",
          int(np.count_nonzero(lengths != urls.str.len().to_numpy())))
else:
    print("\nNo URLLength column found.")
    print("Unified columns:", unified.columns.tolist())

print("\nNo dataset files were changed.")

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import StratifiedGroupKFold

PROJECT_ROOT = Path(
    str(PROJECT_ROOT)
)

RAW_PATH = PROJECT_ROOT / "data/phiusiil.csv"
UNIFIED_PATH = PROJECT_ROOT / "data/phiusiil_unified.csv"
SPLIT_SEED = 42


def file_sha256(path):
    path = portable_recorded_path(path) if "phishing_q1_project" in str(path) else Path(path)
    h = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


# Record exactly which inputs were used.
input_hashes = {
    str(path): file_sha256(path)
    for path in [RAW_PATH, UNIFIED_PATH]
}

raw = pd.read_csv(
    RAW_PATH,
    usecols=["URL", "label"],
    keep_default_na=False
)
unified = pd.read_csv(
    UNIFIED_PATH,
    usecols=["label", "URLLength"],
    keep_default_na=False
)

raw_y = pd.to_numeric(raw["label"], errors="raise").to_numpy()
unified_y = pd.to_numeric(unified["label"], errors="raise").to_numpy()

if len(raw) != len(unified):
    raise ValueError("Source raw/unified row counts differ.")

if not np.isin(raw_y, [0, 1]).all():
    raise ValueError("Source contains missing or unexpected labels.")

if not np.array_equal(raw_y, unified_y):
    raise ValueError("Source raw/unified labels differ.")

if not np.array_equal(
    pd.to_numeric(unified["URLLength"], errors="raise").to_numpy(),
    raw["URL"].astype(str).str.len().to_numpy()
):
    raise ValueError("Source raw/unified URL lengths differ.")

urls = raw["URL"].astype(str).str.strip()

if urls.eq("").any():
    raise ValueError("Blank source URLs require inspection.")

if pd.Series(raw_y).groupby(urls).nunique().gt(1).any():
    raise ValueError("A source URL has conflicting labels.")

y = raw_y.astype(np.int8)

# Every occurrence of the same URL receives the same group identity.
groups, _ = pd.factorize(urls, sort=True)
all_rows = np.arange(len(raw), dtype=np.int64)

# Reserve approximately 20% for source testing.
# Choose the first fold by rule; do not select a fold using model performance.
outer = StratifiedGroupKFold(
    n_splits=5, shuffle=True, random_state=SPLIT_SEED
)
development, test = next(
    outer.split(np.zeros(len(y)), y, groups)
)

# Reserve approximately 20% of the remaining 80% for validation.
# This gives approximately 64% training, 16% validation, 20% testing.
inner = StratifiedGroupKFold(
    n_splits=5, shuffle=True, random_state=SPLIT_SEED
)
train_local, validation_local = next(
    inner.split(
        np.zeros(len(development)),
        y[development],
        groups[development]
    )
)

parts = {
    "source_train": np.sort(development[train_local]),
    "source_validation": np.sort(development[validation_local]),
    "source_test": np.sort(test),
}

# Verify complete row coverage and absence of repeated row assignments.
combined = np.concatenate(list(parts.values()))
if not np.array_equal(np.sort(combined), all_rows):
    raise AssertionError("Partitions do not cover every source row exactly once.")

# Verify URL separation for every pair of partitions.
names = list(parts)
for i, first in enumerate(names):
    for second in names[i + 1:]:
        shared = np.intersect1d(
            groups[parts[first]], groups[parts[second]]
        )
        if shared.size:
            raise AssertionError(f"Shared URLs: {first} / {second}")

summary = []
for name, ix in parts.items():
    if np.unique(y[ix]).size != 2:
        raise AssertionError(f"{name} does not contain both classes.")

    summary.append({
        "partition": name,
        "rows": len(ix),
        "legitimate": int((y[ix] == 0).sum()),
        "phishing": int((y[ix] == 1).sum()),
        "distinct_URLs": int(np.unique(groups[ix]).size),
    })

# Confirm inputs remained unchanged during preparation.
for path, expected_hash in input_hashes.items():
    if file_sha256(path) != expected_hash:
        raise RuntimeError(f"Input changed during preparation: {path}")

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
PREPARATION_DIR = (
    PROJECT_ROOT / "results/revision" / f"url_grouped_preparation_{stamp}"
)
PREPARATION_DIR.mkdir(parents=True, exist_ok=False)

split_path = PREPARATION_DIR / "source_partitions.npz"
np.savez_compressed(split_path, **parts)

record = {
    "seed": SPLIT_SEED,
    "method": "nested StratifiedGroupKFold; first fold at each level",
    "URL_comparison": "strip surrounding whitespace only",
    "row_index": "zero-based positions in the recorded source files",
    "input_sha256": input_hashes,
    "split_file_sha256": file_sha256(split_path),
    "scikit_learn_version": sklearn.__version__,
    "partitions": summary,
    "shared_URLs_between_partitions": 0,
}

(PREPARATION_DIR / "source_partition_record.json").write_text(
    json.dumps(record, indent=2),
    encoding="utf-8"
)

print(pd.DataFrame(summary).to_string(index=False))
print("\nShared URLs between every pair of partitions: 0")
print("\nSaved preparation folder:", PREPARATION_DIR)
print("\nNo dataset files were changed. No models were trained.")

In [ ]:
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd

PROJECT_ROOT = Path(
    str(PROJECT_ROOT)
)

PREPARATION_DIR = PROJECT_ROOT / (
    "results/revision/"
    "url_grouped_preparation_20261020T224241295695Z"
)

SOURCE_RAW = PROJECT_ROOT / "data/phiusiil.csv"
LEGIT_RAW = PROJECT_ROOT / "data/LegitPhish Dataset.csv"
LEGIT_UNIFIED = PROJECT_ROOT / "data/legitphish_unified.csv"


def file_sha256(path):
    path = portable_recorded_path(path) if "phishing_q1_project" in str(path) else Path(path)
    h = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


# Verify that we are using the source inputs and split prepared in Step 3.
source_record = json.loads(
    (PREPARATION_DIR / "source_partition_record.json").read_text()
)
split_path = PREPARATION_DIR / "source_partitions.npz"

if file_sha256(split_path) != source_record["split_file_sha256"]:
    raise ValueError("The saved source partition file has changed.")

for path, expected in source_record["input_sha256"].items():
    if file_sha256(path) != expected:
        raise ValueError(f"A source input has changed: {path}")

target_hashes = {
    str(path): file_sha256(path)
    for path in [LEGIT_RAW, LEGIT_UNIFIED]
}

source = pd.read_csv(
    SOURCE_RAW, usecols=["URL"], keep_default_na=False
)
raw = pd.read_csv(
    LEGIT_RAW, usecols=["URL", "ClassLabel"], keep_default_na=False
)
unified = pd.read_csv(
    LEGIT_UNIFIED, usecols=["label", "URLLength"], keep_default_na=False
)

# Check the corrected raw/unified files before selecting row positions.
native_y = pd.to_numeric(raw["ClassLabel"], errors="raise")
unified_y = pd.to_numeric(unified["label"], errors="raise")

if len(raw) != len(unified):
    raise ValueError("LegitPhish raw/unified row counts differ.")

if not native_y.isin([0, 1]).all():
    raise ValueError("Raw LegitPhish contains missing or unexpected labels.")

# Raw: 1=legitimate, 0=phishing.
# Unified: 0=legitimate, 1=phishing.
y = (1 - native_y).to_numpy(dtype=np.int8)

if not np.array_equal(y, unified_y.to_numpy()):
    raise ValueError("LegitPhish raw/unified labels are not aligned.")

original_urls = raw["URL"].astype(str)
urls = original_urls.str.strip()
source_urls = source["URL"].astype(str).str.strip()

if urls.eq("").any() or source_urls.eq("").any():
    raise ValueError("Blank URLs require inspection.")

if not np.array_equal(
    pd.to_numeric(unified["URLLength"], errors="raise").to_numpy(),
    original_urls.str.len().to_numpy()
):
    raise ValueError("LegitPhish raw/unified URL lengths differ.")

if pd.Series(y).groupby(urls).nunique().gt(1).any():
    raise ValueError("A LegitPhish URL has conflicting labels.")

with np.load(split_path, allow_pickle=False) as saved:
    parts = {name: saved[name].copy() for name in saved.files}

matches = {}
for name in ["source_train", "source_validation", "source_test"]:
    reference_urls = source_urls.iloc[parts[name]]
    matches[name] = urls.isin(reference_urls).to_numpy()

# Preserve the agreed rule: exclude source training AND validation URLs.
eligible = ~(
    matches["source_train"] | matches["source_validation"]
)
selected_rows = np.flatnonzero(eligible).astype(np.int64)
selected_y = y[selected_rows]

# These groups will keep copies of a URL together in target splits.
selected_urls = urls.iloc[selected_rows]
url_groups, _ = pd.factorize(selected_urls, sort=True)

if np.unique(selected_y).size != 2:
    raise ValueError("The selected cohort must contain both classes.")

summary = []
for label, name in [(0, "legitimate"), (1, "phishing")]:
    class_mask = y == label
    summary.append({
        "class": name,
        "corrected_full_rows": int(class_mask.sum()),
        "source_train_matches": int(
            (class_mask & matches["source_train"]).sum()
        ),
        "source_validation_matches": int(
            (class_mask & matches["source_validation"]).sum()
        ),
        "selected_rows": int((class_mask & eligible).sum()),
        "selected_source_test_matches": int(
            (class_mask & eligible & matches["source_test"]).sum()
        ),
    })

# Confirm the inputs did not change during preparation.
for path, expected in {
    **source_record["input_sha256"], **target_hashes
}.items():
    if file_sha256(path) != expected:
        raise RuntimeError(f"Input changed during preparation: {path}")

selection_path = PREPARATION_DIR / "legitphish_selection.npz"
record_path = PREPARATION_DIR / "legitphish_selection_record.json"

if selection_path.exists() or record_path.exists():
    raise FileExistsError(
        "Selection files already exist. Stop here to avoid overwriting them."
    )

np.savez_compressed(
    selection_path,
    corrected_full_row_indices=selected_rows,
    url_group=url_groups.astype(np.int64),
    source_test_match=matches["source_test"][selected_rows],
)

record = {
    "selection_rule": "exclude exact URLs in source training or validation",
    "URL_comparison": "strip surrounding whitespace only",
    "row_index": "zero-based position in corrected full LegitPhish files",
    "input_sha256": target_hashes,
    "source_split_sha256": source_record["split_file_sha256"],
    "selection_file_sha256": file_sha256(selection_path),
    "selected_rows": len(selected_rows),
    "distinct_selected_URLs": int(np.unique(url_groups).size),
    "class_summary": summary,
    "limits": (
        "Source-test URL overlap is counted but retained. "
        "Domain independence has not been established."
    ),
}
record_path.write_text(json.dumps(record, indent=2), encoding="utf-8")

print(pd.DataFrame(summary).to_string(index=False))
print("\nSelected rows:", len(selected_rows))
print("Distinct selected URLs:", np.unique(url_groups).size)
print("Selected matches to source training:",
      int(matches["source_train"][selected_rows].sum()))
print("Selected matches to source validation:",
      int(matches["source_validation"][selected_rows].sum()))
print("Selected matches to source test:",
      int(matches["source_test"][selected_rows].sum()))
print("\nSaved:", selection_path)
print("Saved:", record_path)
print("\nNo dataset CSVs were changed. No models were trained.")

In [ ]:
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd

PROJECT_ROOT = Path(
    str(PROJECT_ROOT)
)
PREPARATION_DIR = PROJECT_ROOT / (
    "results/revision/"
    "url_grouped_preparation_20261020T224241295695Z"
)


def file_sha256(path):
    path = portable_recorded_path(path) if "phishing_q1_project" in str(path) else Path(path)
    h = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


record = json.loads(
    (PREPARATION_DIR / "source_partition_record.json").read_text()
)
split_path = PREPARATION_DIR / "source_partitions.npz"

# Ensure these are the same source inputs and partitions as Step 3.
if file_sha256(split_path) != record["split_file_sha256"]:
    raise ValueError("Saved source partitions have changed.")

for path, expected in record["input_sha256"].items():
    if file_sha256(path) != expected:
        raise ValueError(f"Source input has changed: {path}")

source = pd.read_csv(
    PROJECT_ROOT / "data/phiusiil.csv",
    usecols=["URL", "label"],
    keep_default_na=False
)
target = pd.read_csv(
    PROJECT_ROOT / "data/URL-Phish.csv",
    usecols=["url", "label"],
    keep_default_na=False
)

source_urls = source["URL"].astype(str).str.strip()
target_urls = target["url"].astype(str).str.strip()
source_y = pd.to_numeric(source["label"], errors="raise")
target_y = pd.to_numeric(target["label"], errors="raise")

if (
    not source_y.isin([0, 1]).all()
    or not target_y.isin([0, 1]).all()
):
    raise ValueError("Unexpected or missing labels.")

if source_urls.eq("").any() or target_urls.eq("").any():
    raise ValueError("Blank URLs require inspection.")

# Each source URL must have a consistent label before creating a lookup.
if source_y.groupby(source_urls).nunique().gt(1).any():
    raise ValueError("Conflicting labels within source URLs.")

source_label_lookup = pd.Series(
    source_y.to_numpy(), index=source_urls.to_numpy()
)
source_label_lookup = source_label_lookup[
    ~source_label_lookup.index.duplicated()
]

with np.load(split_path, allow_pickle=False) as saved:
    parts = {name: saved[name].copy() for name in saved.files}

matches = {
    name: target_urls.isin(source_urls.iloc[ix]).to_numpy()
    for name, ix in parts.items()
}

development_match = (
    matches["source_train"] | matches["source_validation"]
)
any_source_match = development_match | matches["source_test"]

matched_source_labels = target_urls.map(source_label_lookup)
conflicts = (
    matched_source_labels.notna()
    & matched_source_labels.ne(target_y)
)

rows = []
for label, name in [(0, "legitimate"), (1, "phishing")]:
    mask = target_y.to_numpy() == label
    rows.append({
        "class": name,
        "full_rows": int(mask.sum()),
        "source_train_matches": int(
            (mask & matches["source_train"]).sum()
        ),
        "source_validation_matches": int(
            (mask & matches["source_validation"]).sum()
        ),
        "source_test_matches": int(
            (mask & matches["source_test"]).sum()
        ),
        "absent_from_train_and_validation": int(
            (mask & ~development_match).sum()
        ),
        "absent_from_all_source": int(
            (mask & ~any_source_match).sum()
        ),
        "conflicting_label_rows": int(
            (mask & conflicts.to_numpy()).sum()
        ),
    })

print(pd.DataFrame(rows).to_string(index=False))
print("\nCounts represent target rows, including repeated URLs.")
print("No files were changed.")

In [ ]:
# Uses the verified target, matches, development_match,
# PROJECT_ROOT, PREPARATION_DIR and file_sha256 from Step 5.

required = [
    "target", "matches", "development_match",
    "PROJECT_ROOT", "PREPARATION_DIR", "file_sha256"
]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        f"Run the Step 5 cell first. Missing variables: {missing}"
    )

raw_path = PROJECT_ROOT / "data/URL-Phish.csv"
unified_path = PROJECT_ROOT / "data/urlphish_unified.csv"

input_hashes = {
    str(path): file_sha256(path)
    for path in [raw_path, unified_path]
}

# Reload the raw file and verify that the Step 5 URL/label rows
# still correspond exactly to the current file.
raw_now = pd.read_csv(
    raw_path, usecols=["url", "label"], keep_default_na=False
)
if not raw_now.equals(target):
    raise ValueError("URL-Phish changed since Step 5. Repeat that check.")

unified = pd.read_csv(
    unified_path,
    usecols=["label", "URLLength"],
    keep_default_na=False
)

y = pd.to_numeric(raw_now["label"], errors="raise").to_numpy()
unified_y = pd.to_numeric(unified["label"], errors="raise").to_numpy()

if not np.isin(y, [0, 1]).all():
    raise ValueError("Missing or unexpected URL-Phish labels.")

if len(raw_now) != len(unified) or not np.array_equal(y, unified_y):
    raise ValueError("Raw/unified counts or labels differ.")

if not np.array_equal(
    pd.to_numeric(unified["URLLength"], errors="raise").to_numpy(),
    raw_now["url"].astype(str).str.len().to_numpy()
):
    raise ValueError("Raw/unified URL lengths differ.")

# Exclude URLs available during source training or validation.
selected_rows = np.flatnonzero(~development_match).astype(np.int64)
selected_urls = (
    raw_now["url"].astype(str).str.strip().iloc[selected_rows]
)

if selected_urls.eq("").any():
    raise ValueError("Blank selected URLs require inspection.")

selected_y = y[selected_rows].astype(np.int8)
if np.unique(selected_y).size != 2:
    raise ValueError("The selection must contain both classes.")

if pd.Series(selected_y).groupby(
    selected_urls.to_numpy()
).nunique().gt(1).any():
    raise ValueError("A selected URL has conflicting labels.")

url_groups, _ = pd.factorize(selected_urls, sort=True)

# Confirm source provenance again before saving.
source_record = json.loads(
    (PREPARATION_DIR / "source_partition_record.json").read_text()
)
for path, expected in {
    **source_record["input_sha256"], **input_hashes
}.items():
    if file_sha256(path) != expected:
        raise RuntimeError(f"Input changed: {path}")

if file_sha256(
    PREPARATION_DIR / "source_partitions.npz"
) != source_record["split_file_sha256"]:
    raise ValueError("Source partitions have changed.")

selection_path = PREPARATION_DIR / "urlphish_selection.npz"
record_path = PREPARATION_DIR / "urlphish_selection_record.json"

if selection_path.exists() or record_path.exists():
    raise FileExistsError(
        "Selection files already exist. Stop to avoid overwriting them."
    )

np.savez_compressed(
    selection_path,
    corrected_full_row_indices=selected_rows,
    url_group=url_groups.astype(np.int64),
    source_test_match=matches["source_test"][selected_rows],
)

selection_record = {
    "selection_rule": "exclude exact URLs in source training or validation",
    "URL_comparison": "strip surrounding whitespace only",
    "row_index": "zero-based position in recorded full URL-Phish files",
    "input_sha256": input_hashes,
    "source_split_sha256": source_record["split_file_sha256"],
    "selection_file_sha256": file_sha256(selection_path),
    "selected_rows": len(selected_rows),
    "legitimate": int((selected_y == 0).sum()),
    "phishing": int((selected_y == 1).sum()),
    "distinct_selected_URLs": int(np.unique(url_groups).size),
    "retained_source_test_matches": int(
        matches["source_test"][selected_rows].sum()
    ),
}
record_path.write_text(
    json.dumps(selection_record, indent=2),
    encoding="utf-8"
)

print("Selected rows:", len(selected_rows))
print("Legitimate:", int((selected_y == 0).sum()))
print("Phishing:", int((selected_y == 1).sum()))
print("Distinct selected URLs:", np.unique(url_groups).size)
print("Source-training matches:",
      int(matches["source_train"][selected_rows].sum()))
print("Source-validation matches:",
      int(matches["source_validation"][selected_rows].sum()))
print("Source-test matches:",
      int(matches["source_test"][selected_rows].sum()))
print("\nSaved:", selection_path)
print("Saved:", record_path)
print("\nNo dataset CSVs were changed. No models were trained.")

In [ ]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED

PROJECT_ROOT = Path(
    str(PROJECT_ROOT)
)

PREPARATION_DIR = PROJECT_ROOT / (
    "results/revision/"
    "url_grouped_preparation_20261020T224241295695Z"
)

filenames = [
    "source_partitions.npz",
    "source_partition_record.json",
    "legitphish_selection.npz",
    "legitphish_selection_record.json",
    "urlphish_selection.npz",
    "urlphish_selection_record.json",
]

missing = [
    name for name in filenames
    if not (PREPARATION_DIR / name).is_file()
]
if missing:
    raise FileNotFoundError(f"Missing preparation files: {missing}")

zip_path = PROJECT_ROOT / "results" / "pipeline_preparation.zip"

with ZipFile(zip_path, "w", compression=ZIP_DEFLATED) as archive:
    for name in filenames:
        archive.write(PREPARATION_DIR / name, arcname=name)

with ZipFile(zip_path, "r") as archive:
    damaged = archive.testzip()
    if damaged is not None:
        raise RuntimeError(f"ZIP verification failed: {damaged}")
    if set(archive.namelist()) != set(filenames):
        raise RuntimeError("ZIP contents do not match the six expected files.")

print("Verified ZIP containing all six preparation files.")
print("Original preparation files and datasets remain unchanged.")

print("Archive saved:", zip_path)
if "google.colab" in sys.modules:
    from google.colab import files
    files.download(str(zip_path))